<a href="https://colab.research.google.com/github/Nurdaylight/Study/blob/main/Text_scraper_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install playwright
!python -m playwright install chromium
!python -m playwright install-deps chromium


In [28]:
from bs4 import BeautifulSoup

# read file containing HTML
with open("novelbin_chapters/input.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# extract all links
links = [a["href"] for a in soup.find_all("a", href=True)]


In [36]:
import asyncio
import os
from playwright.async_api import async_playwright

def clean_lines(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

async def scrape_chapters_from_links(links, delay_sec: float = 1.0):
    os.makedirs("novelbin_chapters", exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
        )

        page = await context.new_page()
        await page.set_extra_http_headers({
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://novelbin.com/",
        })

        for idx, url in enumerate(links):
            print(f"\nScraping ({idx+1}/{len(links)}): {url}")

            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("#chr-content", timeout=30000)

            title = (await page.locator("h2").first.inner_text()).strip()
            raw = await page.locator("#chr-content").inner_text()

            text = clean_lines(raw)

            filename = f"novelbin_chapters/chapter_{links[idx][-10:-1]}.txt"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(title + "\n\n" + text)

            print(f"Saved → {filename}")
            await asyncio.sleep(delay_sec)

        await context.close()
        await browser.close()
i=201
await scrape_chapters_from_links(links[i:i+1], delay_sec=1.2)


Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-201-life
Saved → novelbin_chapters/chapter_r-201-lif.txt


In [23]:
import asyncio
import re
from pathlib import Path
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

START_URL = "https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic"
OUT_DIR = Path("novelbin_chapters")
OUT_DIR.mkdir(exist_ok=True)

def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s_-]+", "-", s)
    return s[:120] if s else "chapter"

def clean_text(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]
    return "\n".join(lines)

def to_me_domain(url: str) -> str:
    return url.replace("https://novelbin.com/", "https://novelbin.me/").replace("http://novelbin.com/", "https://novelbin.me/")

async def ensure_chapter_loaded(page, url: str, timeout: int = 30000) -> None:
    # Try to load URL and wait for chapter content
    await page.goto(url, wait_until="domcontentloaded", timeout=60000)

    try:
        await page.wait_for_selector("#chr-content", state="visible", timeout=timeout)
        return
    except PlaywrightTimeoutError:
        # Debug what we actually loaded
        cur_url = page.url
        title = await page.title()
        html = await page.content()
        print("\n[DEBUG] #chr-content not found on first attempt")
        print("[DEBUG] Loaded URL:", cur_url)
        print("[DEBUG] Page title:", title)
        print("[DEBUG] HTML head snippet:", html[:500].replace("\n", " ") , "...\n")

        # If it looks like a bot-check / interstitial, retry with novelbin.me
        fallback = to_me_domain(url)
        if fallback != url:
            print("[DEBUG] Retrying via:", fallback)
            await page.goto(fallback, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("#chr-content", state="visible", timeout=timeout)
            return

        # Otherwise re-raise
        raise

async def press_next(page, timeout: int = 30000):
    """Use Playwright's trusted keyboard events (D then ArrowRight)."""
    await page.wait_for_selector("#chr-content", state="visible", timeout=timeout)

    old_url = page.url
    old_title = (await page.locator("h2").first.inner_text()).strip()

    # focus in reading area (avoid focus in inputs)
    try:
        await page.locator("#chr-content").click(timeout=5000)
    except:
        await page.click("body")
    await page.evaluate("() => window.focus()")

    async def wait_change():
        try:
            await page.wait_for_url(lambda u: u != old_url, timeout=timeout)
            return True
        except PlaywrightTimeoutError:
            try:
                await page.wait_for_function(
                    """(t) => {
                        const h2 = document.querySelector("h2");
                        return h2 && h2.innerText.trim() !== t;
                    }""",
                    arg=old_title,
                    timeout=timeout,
                )
                return True
            except PlaywrightTimeoutError:
                return False

    # Try D then ArrowRight
    await page.keyboard.press("KeyD")
    moved = await wait_change()
    if not moved:
        await page.keyboard.press("ArrowRight")
        moved = await wait_change()

    if not moved:
        # More debug if navigation fails
        next_href = await page.get_attribute("a#next_chap", "href")
        print("[DEBUG] Key navigation failed; still at:", page.url)
        print("[DEBUG] next_chap href is:", next_href)
        return False

    await page.wait_for_selector("#chr-content", state="visible", timeout=timeout)
    return True

async def scrape_by_keyboard(start_url: str, n: int = 5, delay_sec: float = 0.6):
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
            viewport={"width": 1280, "height": 720},
        )

        page = await context.new_page()

        # IMPORTANT: ensure we actually have a chapter page loaded (with fallback)
        await ensure_chapter_loaded(page, start_url, timeout=30000)

        for i in range(n):
            await page.wait_for_selector("#chr-content", state="visible", timeout=30000)

            title = (await page.locator("h2").first.inner_text()).strip()
            text = clean_text(await page.locator("#chr-content").inner_text())

            filename = OUT_DIR / f"{i+1:04d}-{slugify(title)}.txt"
            filename.write_text(title + "\n\n" + text + "\n", encoding="utf-8")
            print(f"Saved: {filename}")
            print(f"Current URL: {page.url}")  # <- your requested line

            if i == n - 1:
                break

            ok = await press_next(page, timeout=30000)
            if not ok:
                print("Stopping: could not navigate to next chapter via keyboard.")
                break

            await asyncio.sleep(delay_sec)

        await context.close()
        await browser.close()

# Colab/Jupyter
await scrape_by_keyboard(START_URL, n=3, delay_sec=1.0)


Saved: novelbin_chapters/0001-chapter-200-panic.txt
Current URL: https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic


TimeoutError: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("#chr-content") to be visible
